In [ ]:
import os
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "3"


import torch
# torch.cuda.init()
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
!source /home/jupyter/Mrigi/env.sh
hf_token = os.environ.get("HF_TOKEN")

torch.cuda.set_device(0)
import json

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline



3


In [29]:
from pydantic import BaseModel, Field
from typing import Optional, Union
from datetime import datetime
from tqdm import tqdm

In [30]:
!nvidia-smi

Mon Oct 27 17:15:11 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.29.05    Driver Version: 495.29.05    CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A5000    Off  | 00000000:31:00.0 Off |                  Off |
| 30%   32C    P8    20W / 230W |      5MiB / 24256MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A5000    Off  | 00000000:4B:00.0 Off |                  Off |
| 30%   

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [31]:

!kill -9 3772414

/bin/bash: line 0: kill: (3772414) - No such process


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [32]:
# ---------- Step 1: Load and Process the JSON Records ----------
# # Load JSON records from file
# with open('filtered_records_subset_100.json', 'r') as file:
#     filtered_records = json.load(file)

In [33]:
# Process each record: extract paragraphs with "Experimental" supersection,
# concatenate them, and append DOI info.
# documents = []
# metadata = []

In [34]:
# Process records into documents
# documents = []
# metadata = []

# for record in filtered_records:
#     doi = record.get("doi", "Unknown DOI")
    
#     # Add abstract as a separate document
#     abstract_text = record.get("abstract", "").strip()
#     if abstract_text:
#         combined_text = f"{abstract_text}\n\nThis information is from DOI: {doi}"
#         documents.append(combined_text)
#         metadata.append({"doi": doi, "source": "abstract"})
    
#     # Add each paragraph as a separate document
#     for para in record.get("paragraphs", []):
#         paragraph_text = para.get("text", "").strip()
#         if paragraph_text:
#             combined_text = f"{paragraph_text}\n\nThis information is from DOI: {doi}"
#             documents.append(combined_text)
#             metadata.append({"doi": doi, "source": "paragraph"})

# print(f"Total documents processed: {len(documents)}")

In [35]:
# for doc in documents:
#     print(doc)
#     print("-"*40)  # Optional: adds a visual separator between different papers


In [36]:
# ---------- Step 2: Prepare Embeddings for the Existing FAISS Index ----------
# Use the same model configuration that was used when the index was created.
# embeddings = HuggingFaceEmbeddings(model_name="allenai/scibert_scivocab_uncased")
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

In [37]:
# ---------- Step 2: Load the Existing Vector Database from Disk ----------
INDEX_DIRECTORY = Path("faiss_index")
INDEX_NAME = "index"
if not INDEX_DIRECTORY.exists():
    raise FileNotFoundError(f"Vector index directory '{INDEX_DIRECTORY}' not found.")

# Use allow_dangerous_deserialization=True to load indices saved with pickle-based metadata.
vector_db = FAISS.load_local(
    INDEX_DIRECTORY,
    embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=True
)
print(f"Loaded vector database from {INDEX_DIRECTORY} with {vector_db.index.ntotal} vectors")

Loaded vector database from faiss_index with 29300 vectors


In [38]:
# # Print DOIs for up to the first 50 processed documents
# if not metadata:
#     print("No metadata available. Did you load the records?")
# else:
#     for entry in metadata[:50]:
#         print(entry.get("doi", "Unknown DOI"))

In [39]:
# ---------- Step 3: Use the Vector DB in a RAG Pipeline ----------
# Define your query.
# query = "How is hierarchical ZSM-5 synthesized?"
# query = "What is the best way to synthesize ZSM-5 that is hierarchical?"
# query = "How is Silicalite-1 synthesized?"
query = "How can you study the effect of nature of silica source on the purity of template-free ZSM-5?"


In [40]:
# Retrieve top k relevant documents (papers) from the vector database.
retrieved_docs = vector_db.similarity_search(query, k=4)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [41]:
# print(embeddings.embed_query("Silicalite-1 synthesis"))
# print(embeddings.embed_documents(["test text 1", "test text 2"]))


In [42]:
context_text

'Synthesis of ZSM-5 from template-free batches which preceded the preparation of template-free ZSM-5 layers on porous supports was studied to ascertain the effect of nature of silica source on the purity of template-free ZSM-5. Silicic acid and two colloidal silica sols were used as silica sources to prepare the template-free batches with a molar composition of 6.5Na2O:Al2O3:80SiO2:3196H2O. One of the colloidal silica sols contained methanol as stabilizer while the other did not. The product purity and rate of crystallization increased when colloidal silica sols were used as silica source, however, use of silicic acid led to low purity and slow crystallization rate. The methanol in the colloidal silica sol appeared to act as template to promote the crystallization and was occluded in the resultant ZSM-5 pores. The dissolution of the meta-stable ZSM-5 phase and formation of quartz was observed regardless of the nature of the silica source in case of prolonging the crystallization time m

In [43]:
# RAG prompt template
RAG_PROMPT = """
Answer the question based only on the following context:
{context}
Question: {question}
Provide a detailed answer.
Provide which DOI the answer is retrieved from.
"""



In [44]:
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
prompt = rag_prompt.format(context=context_text, question=query)

In [45]:
# ---------- Step 4: Generate an Answer Using an Open-Source LLM ----------
# Setup the open source LLM using Mistral (ensure you have the model or access to it)
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False, use_auth_token=hf_token)

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/models/auto/tokenization_auto.py:655: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [47]:
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("Supported model types:", list(CONFIG_MAPPING.keys()))

Supported model types: ['albert', 'align', 'altclip', 'audio-spectrogram-transformer', 'autoformer', 'bark', 'bart', 'beit', 'bert', 'bert-generation', 'big_bird', 'bigbird_pegasus', 'biogpt', 'bit', 'blenderbot', 'blenderbot-small', 'blip', 'blip-2', 'bloom', 'bridgetower', 'bros', 'camembert', 'canine', 'chinese_clip', 'clap', 'clip', 'clipseg', 'code_llama', 'codegen', 'conditional_detr', 'convbert', 'convnext', 'convnextv2', 'cpmant', 'ctrl', 'cvt', 'data2vec-audio', 'data2vec-text', 'data2vec-vision', 'deberta', 'deberta-v2', 'decision_transformer', 'deformable_detr', 'deit', 'deta', 'detr', 'dinat', 'dinov2', 'distilbert', 'donut-swin', 'dpr', 'dpt', 'efficientformer', 'efficientnet', 'electra', 'encodec', 'encoder-decoder', 'ernie', 'ernie_m', 'esm', 'falcon', 'flaubert', 'flava', 'fnet', 'focalnet', 'fsmt', 'funnel', 'git', 'glpn', 'gpt-sw3', 'gpt2', 'gpt_bigcode', 'gpt_neo', 'gpt_neox', 'gpt_neox_japanese', 'gptj', 'gptsan-japanese', 'graphormer', 'groupvit', 'hubert', 'ibert'

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, use_auth_token=hf_token, torch_dtype=torch.float16)

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/models/auto/auto_factory.py:472: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [49]:
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=10000)

In [50]:
# Wrap the Hugging Face pipeline with LangChain's HuggingFacePipeline interface.
rag_llm = HuggingFacePipeline(pipeline=hf_pipeline)



In [51]:
# Generate the answer based on the prompt that includes retrieved context.
# response_text = rag_llm(prompt)
# print("Response:")
# print(response_text)

In [52]:
def get_unique_dois_with_titles(vector_db, limit=50):
    """Extract unique DOIs and their associated titles from the vector store"""
    seen_dois = set()
    doi_titles = []
    
    # Get all documents
    all_docs = vector_db.similarity_search("", k=vector_db.index.ntotal)
    
    for doc in all_docs:
        # Extract DOI from the document content
        content = doc.page_content
        metadata = doc.metadata
        
        # Find DOI in the content (it's typically at the end of each document)
        if "This information is from DOI:" in content:
            doi = content.split("This information is from DOI:")[-1].strip()
            
            if doi not in seen_dois:
                seen_dois.add(doi)
                # Look for title in metadata or content
                title = metadata.get('title', 'Title not found')
                doi_titles.append({'doi': doi, 'title': title})
                
                if len(seen_dois) >= limit:
                    break
    
    return doi_titles

# Get and display unique DOIs with titles
unique_entries = get_unique_dois_with_titles(vector_db)
print(f"\nFound {len(unique_entries)} unique DOIs:")
for entry in unique_entries:
    print(f"\nDOI: {entry['doi']}")
    print(f"Title: {entry['title']}")
print("\n" + "="*80)


Found 50 unique DOIs:

DOI: 10.1002/ceat.201800386
Title: Title not found

DOI: 10.1007/s11244-012-9837-8
Title: Title not found

DOI: 10.1016/j.powtec.2017.11.053
Title: Title not found

DOI: 10.1002/cctc.201090028
Title: Title not found

DOI: 10.1149/1.1644601
Title: Title not found

DOI: 10.1007/s11051-018-4154-1
Title: Title not found

DOI: 10.1002/cite.201400162
Title: Title not found

DOI: 10.1016/j.fuel.2020.118143
Title: Title not found

DOI: 10.1002/zaac.201000158
Title: Title not found

DOI: 10.1016/j.micromeso.2020.110343
Title: Title not found

DOI: 10.1016/j.cattod.2012.07.032
Title: Title not found

DOI: 10.1016/j.micromeso.2013.12.008
Title: Title not found

DOI: 10.1016/j.applthermaleng.2014.07.002
Title: Title not found

DOI: 10.1149/2.0171816jes
Title: Title not found

DOI: 10.1016/j.jece.2021.106933
Title: Title not found

DOI: 10.1007/bf02065585
Title: Title not found

DOI: 10.1016/j.cattod.2017.05.029
Title: Title not found

DOI: 10.1016/j.ces.2017.02.004
Title: T

In [53]:
# Load the filtered records that contain title information
with open('filtered_records_on_zeolite.json', 'r') as f:
    filtered_records = json.load(f)

# Get DOIs from our vector database
vector_dois = {entry['doi'] for entry in unique_entries}

# Create enriched records matching vector DOIs
enriched_records = []
for record in filtered_records:
    if record.get('doi') in vector_dois:
        enriched_record = {
            'doi': record['doi'],
            'title': record.get('title', 'No title'),
            # 'abstract': record.get('abstract', 'No abstract'),
            # 'journal': record.get('journal', 'No journal')
        }
        enriched_records.append(enriched_record)

# Save the matched records
output_path = Path('zeolite_papers_with_titles.json')
with output_path.open('w') as f:
    json.dump(enriched_records, f, indent=2)

print(f"Saved {len(enriched_records)} matched records with titles to {output_path}")
print("\nFirst few entries:")
for record in enriched_records[:3]:
    print(f"\nTitle: {record['title']}")
    print(f"DOI: {record['doi']}")
    # print(f"Journal: {record['journal']}")
    

Saved 60 matched records with titles to zeolite_papers_with_titles.json

First few entries:

Title: Non-acidic Pd/Y Zeolite Catalysts from Organopalladium Precursors: Preparation and Catalytic Activity in MCP Reforming
DOI: 10.1006/jcat.1994.1275

Title: Adsorption and desorption of propene on a commercial Cu-SSZ-13 SCR catalyst
DOI: 10.1016/j.cattod.2013.10.061

Title: Detailed kinetic modeling of the NH3–NO/NO2 SCR reactions over a commercial Cu-zeolite catalyst for Diesel exhausts after treatment
DOI: 10.1016/j.cattod.2012.09.002


## Evaluating Retrieval Accuracy

We'll use the titles as queries and check if the retrieved documents' DOIs match the ground truth DOIs. We'll calculate:
1. Top-1 accuracy: Does the first retrieved document match?
2. Top-3 accuracy: Do any of the first 3 retrieved documents match?
3. Top-5 accuracy: Do any of the first 5 retrieved documents match?

In [54]:
def evaluate_retrieval_accuracy(records, vector_db, k=5):
    """Evaluate retrieval accuracy using titles as queries.
    Args:
        records: List of dicts containing 'title' and 'doi' pairs
        vector_db: FAISS vector store
        k: Number of documents to retrieve (default=5 for all metrics)
    Returns:
        Dictionary with accuracy metrics
    """
    total = len(records)
    correct_at_1 = 0
    correct_at_3 = 0
    correct_at_5 = 0
    
    print(f"Evaluating {total} queries...")
    for record in tqdm(records):
        query = record['title']
        true_doi = record['doi']
        
        # Get top k retrieved documents
        retrieved_docs = vector_db.similarity_search(query, k=k)
        
        # Extract DOIs from retrieved documents
        retrieved_dois = []
        for doc in retrieved_docs:
            content = doc.page_content
            if "This information is from DOI:" in content:
                doi = content.split("This information is from DOI:")[-1].strip()
                retrieved_dois.append(doi)
        
        # Check accuracy at different k values
        if retrieved_dois and retrieved_dois[0] == true_doi:
            correct_at_1 += 1
        if any(doi == true_doi for doi in retrieved_dois[:3]):
            correct_at_3 += 1
        if any(doi == true_doi for doi in retrieved_dois[:5]):
            correct_at_5 += 1
    
    # Calculate accuracy scores
    accuracy = {
        'top_1': correct_at_1 / total * 100,
        'top_3': correct_at_3 / total * 100,
        'top_5': correct_at_5 / total * 100
    }
    
    return accuracy

In [ ]:
# Run evaluation
accuracy_results = evaluate_retrieval_accuracy(enriched_records, vector_db)

print("Retrieval Accuracy Results:")
print(f"Top-1 Accuracy: {accuracy_results['top_1']:.2f}%")
print(f"Top-3 Accuracy: {accuracy_results['top_3']:.2f}%")
print(f"Top-5 Accuracy: {accuracy_results['top_5']:.2f}%")

NameError: name 'records' is not defined

In [ ]:
import matplotlib.pyplot as plt

# Create bar plot
plt.figure(figsize=(10, 6))
metrics = ['Top-1', 'Top-3', 'Top-5']
values = [accuracy_results['top_1'], accuracy_results['top_3'], accuracy_results['top_5']]

plt.bar(metrics, values, color=['#2ecc71', '#3498db', '#9b59b6'])
plt.title('Retrieval Accuracy at Different K Values')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)

# Add value labels on top of each bar
for i, v in enumerate(values):
    plt.text(i, v + 1, f'{v:.1f}%', ha='center')

plt.show()